<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/PMMHA_Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Clone the repository
!git clone https://github.com/SinaTabakhi/MAGNET.git
%cd MAGNET

# 2. Detect Colab's current PyTorch version dynamically
import torch
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')
print(f"Colab is using Torch {torch_version} with CUDA {cuda_version}")

# 3. Install PyG binaries that match Colab's environment exactly
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html
!pip install torch-geometric==2.4.0

# 4. Install the remaining requirements
!pip install lightning==2.1.3 pandas matplotlib umap-learn yacs comet_ml "ray[tune]"

Cloning into 'MAGNET'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 202 (delta 8), reused 0 (delta 0), pack-reused 167 (from 1)
Receiving objects: 100% (202/202), 53.31 MiB | 26.92 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Updating files: 100% (151/151), done.
/content/MAGNET
Colab is using Torch 2.11.0 with CUDA 128
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 108.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 128.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 117.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [3]:
%cd /content/MAGNET

/content/MAGNET


In [17]:
!python main_inference.py --cfg configs/MAGNET_OV.yaml

Epoch 9: 100% 1/1 [00:00<00:00,  5.20it/s, v_num=2, train_kl_loss=0.0343, train_cls_loss=0.646, train_total_loss=0.649, train_acc=0.582, train_auroc=0.669, train_auprc=0.657, train_f1=0.545, train_mcc=0.162]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation:   0% 0/1 [00:00<?, ?it/s]        
Validation DataLoader 0:   0% 0/1 [00:00<?, ?it/s]
Validation DataLoader 0: 100% 1/1 [00:00<00:00, 2601.93it/s]
Epoch 10: 100% 1/1 [00:00<00:00,  5.26it/s, v_num=2, train_kl_loss=0.0329, train_cls_loss=0.639, train_total_loss=0.642, train_acc=0.627, train_auroc=0.693, train_auprc=0.654, train_f1=0.630, train_mcc=0.256]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation:   0% 0/1 [00:00<?, ?it/s]        
Validation DataLoader 0:   0% 0/1 [00:00<?, ?it/s]
Validation DataLoader 0: 100% 1/1 [00:00<00:00, 2513.06it/s]
Epoch 11: 100% 1/1 [00:00<00:00,  4.40it/s, v_num=2, train_kl_loss=0.0338, train_cls_loss=0.608, train_total_loss=0.611, train_acc=0.655, train_auroc=0.751, train_auprc=0.743,

In [ ]:
# base_models.py code[as the repo clone is temporary]
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import glorot
from torch_geometric.nn.conv import MessagePassing
from torch import Tensor


class MLPEncoder(nn.Module):
    def __init__(self, in_dims, hid_dims, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.encoder_layers = nn.Sequential(
            Linear(in_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p=dropout_rate),
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros')
        )

    def forward(self, x):
        x = self.encoder_layers(x)
        return x
# fuse() function_______________________________________________________________
def fuse(x_proj, mask, mode="original", att_lin=None, shared_weights=None):
    """
    Standalone fusion function for MAGNET Ablation Study.
    """
    batch_size, num_heads, num_modalities, head_dims = x_proj.size()

    if mode == "original":
        assert att_lin is not None, "att_lin parameter is required for 'original' mode"
        att_scores = torch.matmul(x_proj, att_lin.transpose(-1, -2)).squeeze(-1)
        att_scores = att_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(att_scores, dim=-1)

    elif mode == "equal":
        counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
        equal_weights = mask / counts
        att_weights = equal_weights.unsqueeze(1).expand(-1, num_heads, -1)

    elif mode == "shared":
        assert shared_weights is not None, "shared_weights parameter is required for 'shared' mode"
        shared_scores = shared_weights.view(1, 1, num_modalities).expand(batch_size, num_heads, -1)
        shared_scores = shared_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(shared_scores, dim=-1)

    else:
        raise ValueError(f"Unknown mode: {mode}. Choose 'equal', 'original', or 'shared'.")

    att_weights = att_weights * mask.unsqueeze(1)
    fused_embeddings = torch.sum(att_weights.unsqueeze(-1) * x_proj, dim=2)

    return fused_embeddings, att_weights
#_______________________________________________________________________________
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dims, num_heads, num_modalities=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dims = hid_dims // num_heads
        assert (
            self.head_dims * num_heads == hid_dims
        ), "hid_dims must be divisible by num_heads"

        self.lin_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')
        self.att_lin = nn.Parameter(torch.empty(num_heads, 1, self.head_dims))
        self.out_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')

        # Register the shared learnable weights for Mode C here
        self.shared_weights = nn.Parameter(torch.zeros(num_modalities))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_proj.reset_parameters()
        self.out_proj.reset_parameters()
        glorot(self.att_lin)
        nn.init.normal_(self.shared_weights, mean=0.0, std=0.1)

    def forward(self, x, mask, mode="original"):
        """
        x: [batch_size, num_modalities, hid_dims] - Patient embeddings for all modalities
        mask: [batch_size, num_modalities] - Mask indicating available modalities
        """
        batch_size, num_modalities, hid_dims = x.size()

        # 1. Linear projection
        x_proj = self.lin_proj(x).view(batch_size, num_modalities, self.num_heads, self.head_dims)
        x_proj = x_proj.permute(0, 2, 1, 3)

        # ─── FORCE ACTIVE MODE HERE ───────────────────────────────────────
        current_mode = "shared"  # Ensure this is set to "shared"
        # ───────────────────────────────────────────────────────────────────

        # ─── FORCE UNIQUE SHARED WEIGHTS ──────────────────────────────────
        if current_mode == "shared":
            # This forces a fixed global bias: heavily favoring DNA over mRNA and miRNA
            with torch.no_grad():
                self.shared_weights.copy_(torch.tensor([2.5, -1.0, -1.5], device=x.device))
        # ───────────────────────────────────────────────────────────────────

        if not getattr(self, '_has_printed_mode', False):
            print(f"\n[MAGNET EXECUTION] >>> Current Fusion Mode Active: {current_mode.upper()} <<<\n")
            if current_mode == "shared":
                probs = torch.softmax(self.shared_weights, dim=0).detach().cpu().numpy()
                print(f"[FORCED STATIC BIAS] DNA: {probs[0]:.4f}, mRNA: {probs[1]:.4f}, miRNA: {probs[2]:.4f}\n")
            self._has_printed_mode = True

        # 2. Call standalone fuse function
        fused_embeddings, att_weights = fuse(
            x_proj=x_proj,
            mask=mask,
            mode=current_mode,
            att_lin=self.att_lin,
            shared_weights=self.shared_weights
        )

        # 3. Concatenate heads and project output
        fused_embeddings = fused_embeddings.view(batch_size, -1)
        output = self.out_proj(fused_embeddings)

        return output, att_weights


class EdgeSAGEConv(MessagePassing):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        aggr = "mean",
        bias: bool = True,
        edge_dim: int = None,
        **kwargs,
    ):
        super().__init__(aggr, **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        in_channels = (in_channels, in_channels)

        if self.edge_dim is not None:
            self.lin_msg = Linear(in_channels[0] + self.edge_dim, in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)

        self.lin = Linear(in_channels[0], in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)
        self.lin_l = Linear(in_channels[0], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=bias)
        self.lin_r = Linear(in_channels[1], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=False)

        self.act_msg = nn.ReLU()

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()
        if self.edge_dim is not None:
            self.lin_msg.reset_parameters()

    def forward(self, x, edge_index, edge_attr = None):
        if isinstance(x, Tensor):
            x = (x, x)

        x = (self.lin(x[0]).relu(), x[1])

        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.lin_l(out)
        x_r = x[1]
        out = out + self.lin_r(x_r)

        return out

    def message(self, x_j, edge_attr = None):
        if edge_attr is not None and self.edge_dim is not None:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(-1)
            msg = torch.cat([x_j, edge_attr], dim=-1)
            return self.act_msg(self.lin_msg(msg))
        return x_j


class GNNDecoder(nn.Module):
    def __init__(self, hid_dims, out_dims, num_layers, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.negative_slope = negative_slope
        self.conv_layers = nn.ModuleList()

        for _ in range(num_layers):
            self.conv_layers.append(EdgeSAGEConv(hid_dims, hid_dims, edge_dim=1))

        self.decoder_layers = nn.Sequential(
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        self.final_layer = Linear(hid_dims, out_dims, weight_initializer='glorot', bias_initializer='zeros')


    def forward(self, x, edge_index, edge_attr=None, return_embedding=False):
        for conv in self.conv_layers:
            x = conv(x, edge_index, edge_attr=edge_attr)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)

        x = self.decoder_layers(x)
        embeddings = x

        logits = self.final_layer(x)

        if return_embedding:
            return logits, embeddings

        return logits